In [2]:
from pyomo.environ import (
    ConcreteModel,
    Objective,
    Expression,
    value,
    Var,
    Param,
    Constraint,
    Set,
    Var,
    Block,
    log,
    log10,
    units as pyunits,
    NonNegativeReals,
)

from pyomo.network import Port
from idaes.core import FlowsheetBlock
from idaes.core.solvers.get_solver import get_solver

from idaes.core.util.model_statistics import *
from idaes.core.util.scaling import *

from idaes.core import FlowsheetBlock, UnitModelCostingBlock

from watertap.core.zero_order_costing import ZeroOrderCosting
from watertap.core.util.model_diagnostics.infeasible import *
from watertap.costing import WaterTAPCosting


from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from idaes.core.util.model_statistics import (
    degrees_of_freedom,
    number_variables,
    number_total_constraints,
    number_unused_variables,
)

from IPython.display import clear_output
from copy import deepcopy
from watertap.property_models.seawater_prop_pack import SeawaterParameterBlock
from watertap.property_models.water_prop_pack import WaterParameterBlock
from watertap_contrib.seto.costing import  SETOWaterTAPCosting #SETOZeroOrderCosting,

from watertap_contrib.seto.costing import (
    TreatmentCosting,
    EnergyCosting,
    SETOSystemCosting,
)

# from watertap_contrib.seto.solar_models.zero_order import PhotovoltaicZO
# from watertap_contrib.seto.energy import solar_energy
# from watertap_contrib.seto.core import SETODatabase, PySAMWaterTAP
from watertap_contrib.seto.unit_models.zero_order.chemical_softening_zo import ChemicalSofteningZO

from watertap_contrib.seto.property_models.basic_water_properties import (
    BasicWaterParameterBlock,
)

In [173]:
component_list = ["Ca_2+","Alkalinity_2-","TDS",]

m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)
m.fs.properties =  BasicWaterParameterBlock(solute_list=component_list)

m.fs.soft = soft = ChemicalSofteningZO(
    property_package=m.fs.properties, silica_removal= False,
    softening_procedure_type= 'single_stage_lime'
)

m.fs.properties.calculate_scaling_factors()

prop_in = soft.properties_in[0]
prop_out = soft.properties_out[0]
prop_waste = soft.properties_waste[0]

print(f"DOF = {degrees_of_freedom(m)}")


q_in = 10/0.99* 3785 * pyunits.m**3 / pyunits.day  # m3/d

ca_in = 1* pyunits.kg / pyunits.m**3
alk_in = 0.4* pyunits.kg / pyunits.m**3 
tds_in = 10 * pyunits.kg / pyunits.m**3 
co2_in  = 0.10844915 * pyunits.kg / pyunits.m**3 


prop_in.conc_mass_comp["Ca_2+"].fix(ca_in)
prop_in.conc_mass_comp["Alkalinity_2-"].fix(alk_in)
prop_in.conc_mass_comp["TDS"].fix(tds_in)

prop_in.flow_vol.fix(q_in)

soft.no_of_mixer.fix(1)
soft.no_of_floc.fix(2)

soft.retention_time_mixer.fix(0.4)
soft.retention_time_floc.fix(25)
soft.retention_time_sed.fix(130)
soft.retention_time_recarb.fix(20)
soft.sedimentation_overflow.fix()

soft.vel_gradient_mix.fix(300)
soft.vel_gradient_floc.fix(50)

soft.frac_vol_recovery.fix()
soft.removal_efficiency.fix(0)

soft.CO2_CaCO3.fix(co2_in)

soft.excess_CaO.fix(0)
soft.CO2_second_basin.fix(0)
soft.MgCl2_dosing.fix(0)
soft.Na2CO3_dosing.fix(0)

soft.initialize()

m.fs.costing = TreatmentCosting()
soft.costing = UnitModelCostingBlock(flowsheet_costing_block = m.fs.costing)
m.fs.costing.cost_process()
m.fs.costing.add_LCOW(prop_in.flow_vol)

solver = get_solver()
results = solver.solve(m)

print(f"DOF = {degrees_of_freedom(m)}")

DOF = 17
2023-07-20 15:29:38 [INFO] idaes.init.fs.soft: Initialization Step 1a Complete.
2023-07-20 15:29:38 [INFO] idaes.init.fs.soft.properties_out: State Released.
2023-07-20 15:29:38 [INFO] idaes.init.fs.soft: Initialization Step 1b Complete.
2023-07-20 15:29:38 [INFO] idaes.init.fs.soft.properties_waste: State Released.
2023-07-20 15:29:38 [INFO] idaes.init.fs.soft: Initialization Step 1c Complete.
2023-07-20 15:29:38 [INFO] idaes.init.fs.soft: Initialization Step 2 optimal - Optimal Solution Found.
2023-07-20 15:29:38 [INFO] idaes.init.fs.soft.properties_in: State Released.
2023-07-20 15:29:38 [INFO] idaes.init.fs.soft: Initialization Complete: optimal - Optimal Solution Found
2023-07-20 15:29:38 [WARNING] idaes.core.base.costing_base: fs.soft capital_cost component has a lower bound less than zero. Be aware that this may result in negative costs during optimization.
2023-07-20 15:29:38 [WARNING] idaes.core.base.costing_base: fs.soft fixed_operating_cost component has a lower bou